# **GPU Environment Verfication**

In [ ]:
import torch

In [ ]:
# Display driver version, CUDA version, and available VRAM.
# Confirm the Colab runtime has a GPU before proceeding.

!nvidia-smi

In [ ]:
# Verify the PyTorch build recognises the CUDA runtime.
# If CUDA is not available the training script will fail at device placement.

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Print the GPU model and total VRAM.
# ViT-L fine-tuning requires at least ~16 GB; A100 (40/80 GB) is recommended.

if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# **Dependencies Installation**

In [ ]:
import importlib

In [ ]:
# --- System dependencies ---
# ffmpeg + libav* are required by decord to decode EK-100 video clips at runtime.

!apt-get install -qq ffmpeg libavcodec-dev libavformat-dev libavutil-dev libswscale-dev

# --- PyTorch (pinned) ---
# Both AVION and flash-attn require torch 2.4.1 built against CUDA 12.1.
# Upgrading torch without rebuilding flash-attn will break the install.

!pip install torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --index-url https://download.pytorch.org/whl/cu121

# --- Core Python dependencies ---
# einops: tensor reshaping | kornia: image augmentations | timm: ViT backbone
# transformers: text encoder | decord: fast video decoding | ninja: C++ JIT builder

!pip install -q einops kornia timm transformers decord ninja

# --- CLIP ---
# OpenAI CLIP for text encoding; open_clip_torch for the open-source variant.
# reranking provides mAP / nDCG evaluation helpers.

!pip install -q git+https://github.com/openai/CLIP.git
!pip install open_clip_torch reranking

# --- Flash Attention ---
# Must be installed LAST, after the pinned torch/CUDA above (compiles from source, ~5 min).
# flash-attn halves attention memory and significantly speeds up ViT-L training.

!pip install flash-attn

In [ ]:
# Confirm flash_attn was installed correctly.
# If False, training still runs but is slower and consumes more VRAM.

HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
print(f"flash_attn available: {HAS_FLASH_ATTN}")
print("Done.")

# **Google Drive Mount**

In [ ]:
from google.colab import drive

In [ ]:
# Mount Google Drive so that datasets, checkpoints, and experiment outputs
# persist across Colab sessions. force_remount ensures a clean mount even
# if the session was interrupted mid-run.

drive.mount("/content/drive", force_remount = True)

# **Base Fine-tuning**

In [ ]:
import os

In [ ]:
# Change into the cloned SMS-Loss repo on Drive so that torchrun can resolve
# relative script paths (e.g. scripts/ammplus_finetune.py).

os.chdir("/content/drive/MyDrive/SMS_Loss_Custom")

In [ ]:
# All project paths are derived from GDRIVE so a single change propagates everywhere.
# ANNOT_DIR — EK-100 retrieval CSVs and relevancy pickles
# DATA_ROOT  — Pre-processed EK-100 video clips (320p, 15-sec chunks, 30 fps)
# EXP_DIR    — Checkpoints, logs, and evaluation outputs (persisted to Drive)
# PRETRAIN_CKPT — AVION ViT-L LaViLa pretrain weights (base for fine-tuning)

GDRIVE    = "/content/drive/MyDrive"
ANNOT_DIR = f"{GDRIVE}/EK100_annotations"
DATA_ROOT = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264"
EXP_DIR   = f"{GDRIVE}/experiments/sms_vitl"
PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"

# Annotation files for the EK-100 retrieval benchmark
TRAIN_CSV = f"{ANNOT_DIR}/EPIC_100_retrieval_train.csv"
TEST_CSV  = f"{ANNOT_DIR}/EPIC_100_retrieval_test.csv"
TRAIN_REL = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_train.pkl"
TEST_REL  = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

In [ ]:
# Build the optional flash-attn CLI flag.
# The flag is passed to the training script only when flash_attn is available;
# omitting it falls back to standard attention (slower, higher VRAM usage).

HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
flash_flag = "--use-flash-attn" if HAS_FLASH_ATTN else ""
print(f"flash_attn: {'enabled' if HAS_FLASH_ATTN else 'disabled'}")

In [ ]:
# Launch the base fine-tuning run from the AVION pretrain checkpoint.
# Key hyperparameters:
#   --batch-size 48     : per-GPU batch size (adjust down if OOM)
#   --epochs 10         : first training round; resume script extends this
#   --lr 2e-5           : learning rate for the SMS-loss fine-tuning stage
#   --loss-margin 0.6   : SMS margin hyperparameter (theta in the paper)
#   --loss-thres  0.1   : relevancy threshold below which pairs are ignored
#   --grad-checkpointing: trades compute for VRAM to fit ViT-L on a single GPU

cmd = f"""\
torchrun --nproc_per_node=1 scripts/ammplus_finetune.py \\
  --root "{DATA_ROOT}" \\
  --train-metadata "{TRAIN_CSV}" \\
  --val-metadata   "{TEST_CSV}" \\
  --relevancy-train "{TRAIN_REL}" \\
  --relevancy-test  "{TEST_REL}" \\
  --pretrain-model  "{PRETRAIN_CKPT}" \\
  --model CLIP_VITL14 \\
  --batch-size 48 \\
  --update-freq 1 \\
  --epochs 10 \\
  --lr 2e-5 \\
  --loss-margin 0.6 \\
  --loss-thres  0.1 \\
  --use-fast-conv1 \\
  {flash_flag} \\
  --grad-checkpointing \\
  --output-dir "{EXP_DIR}"
"""

In [ ]:
print(cmd)
!{cmd}

# **Resume Fine-tuning from Checkpoint**

In [ ]:
import os

In [ ]:
# Change into the SMS-Loss repo root so torchrun resolves relative script paths.

os.chdir("/content/drive/MyDrive/SMS_Loss_Custom")

In [ ]:
# Re-declare all paths in case this section is run independently of the base
# fine-tuning section above (e.g. after a Colab session restart).

GDRIVE    = "/content/drive/MyDrive"
ANNOT_DIR = f"{GDRIVE}/EK100_annotations"
DATA_ROOT = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264"
EXP_DIR   = f"{GDRIVE}/experiments/sms_vitl"
PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"

TRAIN_CSV = f"{ANNOT_DIR}/EPIC_100_retrieval_train.csv"
TEST_CSV  = f"{ANNOT_DIR}/EPIC_100_retrieval_test.csv"
TRAIN_REL = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_train.pkl"
TEST_REL  = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

In [ ]:
# Re-check flash-attn availability (needed if this section runs after a kernel restart).

HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
flash_flag = "--use-flash-attn" if HAS_FLASH_ATTN else ""
print(f"flash_attn: {'enabled' if HAS_FLASH_ATTN else 'disabled'}")

In [ ]:
# Path to the checkpoint saved at the end of Round 1 (base fine-tuning, epoch 10).
# Update this to point to a different checkpoint to resume from a different epoch.

CKPT = f"{EXP_DIR}/checkpoint_round_1.pt"

In [ ]:
# Resume fine-tuning from a previously saved checkpoint.
# Key differences from the base run:
#   --pretrain-model CKPT : loads weights from the Round 1 checkpoint
#   --resume         CKPT : restores optimizer state and epoch counter
#   --pretrain-zoo openai : tells the model which CLIP weight space to expect
#   --epochs 50 / --start-epoch 10 : continues from epoch 10 up to epoch 50

cmd = f"""\
torchrun --nproc_per_node=1 scripts/ammplus_finetune.py \\
  --root "{DATA_ROOT}" \\
  --train-metadata "{TRAIN_CSV}" \\
  --val-metadata   "{TEST_CSV}" \\
  --relevancy-train "{TRAIN_REL}" \\
  --relevancy-test  "{TEST_REL}" \\
  --pretrain-model  "{CKPT}" \\
  --resume          "{CKPT}" \\
  --pretrain-zoo    openai \\
  --model CLIP_VITL14 \\
  --batch-size 48 \\
  --update-freq 1 \\
  --epochs 50 \\
  --start-epoch 10 \\
  --lr 2e-5 \\
  --loss-margin 0.6 \\
  --loss-thres  0.1 \\
  --use-fast-conv1 \\
  {flash_flag} \\
  --grad-checkpointing \\
  --output-dir "{EXP_DIR}"
"""

In [ ]:
print(cmd)
!{cmd}

# **Inference over Test**

In [ ]:
import os

In [ ]:
# Change into the SMS-Loss repo root so torchrun resolves relative script paths.

os.chdir("/content/drive/MyDrive/SMS_Loss_Custom")

In [ ]:
# Re-declare all paths in case this section is run independently of the
# training sections above (e.g. after a Colab session restart).

GDRIVE    = "/content/drive/MyDrive"
ANNOT_DIR = f"{GDRIVE}/EK100_annotations"
DATA_ROOT = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264"
EXP_DIR   = f"{GDRIVE}/experiments/sms_vitl"
PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"

TRAIN_CSV = f"{ANNOT_DIR}/EPIC_100_retrieval_train.csv"
TEST_CSV  = f"{ANNOT_DIR}/EPIC_100_retrieval_test.csv"
TRAIN_REL = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_train.pkl"
TEST_REL  = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

In [ ]:
# Re-check flash-attn availability (needed if this section runs after a kernel restart).

HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
flash_flag = "--use-flash-attn" if HAS_FLASH_ATTN else ""
print(f"flash_attn: {'enabled' if HAS_FLASH_ATTN else 'disabled'}")

In [ ]:
# Checkpoint to evaluate. Point this to any saved .pt file from the experiment dir.

CKPT = f"{EXP_DIR}/checkpoint_round_1.pt"

In [ ]:
# Run inference on the EK-100 retrieval test split and produce submission.pkl.
# Key flags:
#   --project-embed-dim 256 : final embedding dimension after the projection head
#   --flip                  : test-time horizontal flip augmentation
#   --clip-length 32        : number of frames sampled per video clip
#   --batch-size 32         : per-GPU batch size for the forward pass
# Outputs submission.pkl (sim_mat + id lists) to EXP_DIR.

cmd = f"""\
torchrun --nproc_per_node=1 scripts/test_mir.py \\
  --root "{DATA_ROOT}" \\
  --train-metadata "{TRAIN_CSV}" \\
  --val-metadata   "{TEST_CSV}" \\
  --pretrain-model "{CKPT}" \\
  --model CLIP_VITL14 \\
  --project-embed-dim 256 \\
  --use-fast-conv1 \\
  {flash_flag} \\
  --flip \\
  --clip-length 32 \\
  --batch-size 32 \\
  --output-dir "{EXP_DIR}"
"""

In [ ]:
print(cmd)
!{cmd}

# **Build Submission Zip**

In [ ]:
import os, pickle, subprocess
import numpy as np

In [ ]:
# Load the raw output produced by test_mir.py.
# submission.pkl contains:
#   sim_mat  — (N_vis x N_txt) float similarity matrix
#   vis_ids  — list of video segment IDs (rows of sim_mat)
#   txt_ids  — list of caption / narration IDs (columns of sim_mat)

pkl_path = f"{EXP_DIR}/submission.pkl"
with open(pkl_path, "rb") as f:
    raw_out = pickle.load(f)

sim_mat = np.array(raw_out["sim_mat"], dtype=np.float32)
vis_ids = [str(v) for v in raw_out["vis_ids"]]
txt_ids = [str(t) for t in raw_out["txt_ids"]]

print(f"sim_mat : {sim_mat.shape}  dtype={sim_mat.dtype}")
print(f"vis_ids : {len(vis_ids)}  |  txt_ids : {len(txt_ids)}")

# SLS (Supervision Level Score) fields required by the Codabench grader.
# Adjust these values to match the actual supervision used:
#   SLS_PT  — pre-training supervision level (0–5)
#   SLS_TL  — training label supervision level (0–5)
#   SLS_TD  — training data supervision level  (0–5)

SLS_PT = 2
SLS_TL = 3
SLS_TD = 3

In [ ]:
def make_compat_pickle(sim_mat, vis_ids, txt_ids, sls_pt, sls_tl, sls_td):

    """Serialize the submission payload as a protocol-2 pickle.

    The Codabench grader runs an older numpy version that expects the legacy
    'numpy.core.multiarray' module path. numpy >= 2.0 emits
    'numpy._core.multiarray' instead, which causes an UnpicklingError on the
    grader side. The byte-level replacement below patches the serialized data
    so it loads correctly on both old and new numpy versions.
    """

    payload = {
        "version":   "0.1",
        "challenge": "multi_instance_retrieval",
        "sls_pt":    sls_pt,
        "sls_tl":    sls_tl,
        "sls_td":    sls_td,
        "sim_mat":   sim_mat,
        "vis_ids":   vis_ids,
        "txt_ids":   txt_ids,
    }
    
    raw = pickle.dumps(payload, protocol=2)
    # Patch numpy >= 2.0 private module path → legacy public path for grader compatibility.
    raw = raw.replace(b"numpy._core.multiarray", b"numpy.core.multiarray")
    return raw

In [ ]:
# Serialize the submission payload, verify it round-trips correctly,
# then zip it into the final submission archive.

tmp_pkl = "/tmp/test.pkl"
pkl_bytes = make_compat_pickle(sim_mat, vis_ids, txt_ids, SLS_PT, SLS_TL, SLS_TD)
with open(tmp_pkl, "wb") as f:
    f.write(pkl_bytes)

# Round-trip check: re-load and confirm the similarity matrix shape is preserved.

check = pickle.loads(pkl_bytes)
assert np.array(check["sim_mat"]).shape == sim_mat.shape, "Shape mismatch after re-load!"
print(f"test.pkl OK — {len(pkl_bytes)/1e6:.1f} MB")

# Package test.pkl into a zip — the Codabench grader expects exactly this layout.

zip_path = f"{EXP_DIR}/submission.zip"
subprocess.run(
    f'cd /tmp && zip -j "{zip_path}" test.pkl',
    shell=True, check=True, capture_output=True
)

print(f"\nReady: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("Submit to: https://www.codabench.org/competitions/12008/")